# IndicConformer 600M — Hindi Fine-Tuning Pipeline

**Project:** Offline Multilingual Indian Speech-to-Speech Communication
**Goal:** Fine-tune AI4Bharat IndicConformer 600M for Hindi on Common Voice, then evaluate against a frozen 3,342-sample test set (baseline WER = 16.72%).

---

## Pipeline Index

| # | Section | Purpose |
|---|---------|---------|
| 1 | Environment | Verify GPU, Python, PyTorch, CUDA |
| 2 | Repository | Clone AI4Bharat/NeMo (nemo-v2) |
| 3a | Dependencies: Install | pip install pinned packages |
| 3b | Dependencies: Verify | Import NeMo, PL, torchmetrics after runtime restart |
| 4 | Checkpoint | Download / verify IndicConformer.nemo (~2.35 GB) |
| 5 | Dataset Paths | Locate train.tsv, dev.tsv, test.tsv, clips/ |
| 6 | Dataset Validation | Audit audio files, transcripts, durations |
| 7 | Manifest Generation | NeMo JSON-lines manifests + smoke-test subsets |
| 8 | Model Loading | Restore .nemo, inspect config, configure for CTC fine-tuning |
| 9 | Smoke Test | 5-step sanity run: dataloader, GPU, forward/backward, optimizer |
| 10 | Full Fine-Tuning | Hindi CTC adaptation (AdamW, AMP FP16, grad accumulation) |
| 11 | Checkpoint Saving | Save fine-tuned .nemo to separate output directory |
| 12 | Evaluation | Run on frozen 3,342-sample test set |
| 13 | Comparison | Baseline (16.72%) vs fine-tuned WER, latency, RTF |
| 14 | Results | Qualitative inspection + next steps |

> **IMPORTANT:** After running Section 3a, you **must** restart the runtime before running Section 3b.


In [1]:
# 1. 安装 NeMo 的核心依赖项
!pip install hydra-core pytorch-lightning omegaconf

# 2. 如果你是从源码（Local Folder）安装的 NeMo，需要通过 -e 包含所有依赖项
!pip install -e .

# 3. 或者直接通过 PyPI 安装官方稳定版
!pip install nemo_toolkit[asr]

import sys
import os
import site
import pkgutil
import importlib.util

print("=" * 60)
print("   SECTION 3b: VERIFYING IMPORTS WITH PYTHON 3.12 PATCHES")
print("=" * 60)

# ── 1. CORE PATH AND SYSTEM ENVIRONMENT SETUP ───────────────────
NEMO_DIR = "/content/NeMo"
if NEMO_DIR not in sys.path:
    sys.path.insert(0, NEMO_DIR)

# ── 2. ADVANCED PYTHON 3.12 COMPATIBILITY INJECTIONS ───────────
# Stub out the completely removed 'imp' module which old setuptools/lightning dependencies require
if "imp" not in sys.modules:
    try:
        spec = importlib.util.spec_from_loader("imp", loader=None)
        mock_imp = importlib.util.module_from_spec(spec)
        mock_imp.find_module = lambda *args, **kwargs: (None, None, None)
        sys.modules["imp"] = mock_imp
        print("Successfully injected dummy 'imp' module into sys.modules.")
    except Exception as e:
        print(f"Failed to inject dummy imp: {e}")

# Stub out the removed pkgutil.ImpImporter class
if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = type("ImpImporter", (), {})
    print("Patched pkgutil.ImpImporter missing attribute.")

# Patch setuptools namespace system before lightning imports it
try:
    import pkg_resources
    pkg_resources.declare_namespace = lambda name: None
    print("Patched pkg_resources.declare_namespace to no-op.")
except Exception:
    pass

# ── 3. PYTORCH LIGHTNING v2+ NEPTUNE LOGGER COMPATIBILITY PATCH ──
# Dynamically register a dummy NeptuneLogger inside pytorch_lightning to stop NeMo from crashing
try:
    import pytorch_lightning as pl
    import pytorch_lightning.loggers as pl_loggers
    
    if not hasattr(pl_loggers, "NeptuneLogger"):
        # Create a dummy class stub
        class DummyNeptuneLogger:
            def __init__(self, *args, **kwargs):
                raise NotImplementedError(
                    "NeptuneLogger was mocked for import compatibility. "
                    "If you need to use Neptune tracking, please run: pip install neptune"
                )
        
        # Inject the mock directly into the loggers module
        pl_loggers.NeptuneLogger = DummyNeptuneLogger
        sys.modules["pytorch_lightning.loggers"].NeptuneLogger = DummyNeptuneLogger
        print("Injected compatibility stub for NeptuneLogger into pytorch_lightning.loggers.")
except Exception as e:
    print(f"Warning during Neptune Logger injection: {e}")

print("-" * 60)

# ── 4. EXECUTE FRAMEWORK IMPORTS ────────────────────────────────
try:
    import torch
    print(f"PyTorch          : {torch.__version__}")

    import torchmetrics
    print(f"torchmetrics     : {torchmetrics.__version__}")

    print(f"pytorch-lightning: {pl.__version__}")

    # Now importing NeMo will bypass the missing NeptuneLogger check safely
    import nemo
    import nemo.collections.asr as nemo_asr
    from nemo.collections.asr.models import ASRModel
    print(f"NeMo             : {nemo.__version__}")
    print(f"NeMo ASR         : OK")

    import transformers
    import huggingface_hub
    import datasets
    print(f"transformers     : {transformers.__version__}")
    print(f"huggingface_hub  : {huggingface_hub.__version__}")
    print(f"datasets         : {datasets.__version__}")

    print("=" * 60)
    print("All imports verified successfully.")
    print("=" * 60)

except ImportError as e:
    print("\n[IMPORT ERROR DETECTED]")
    print(e)
    print("\nIf errors persist, you might need to install 'neptune' to naturally satisfy the dependency:")
    print("!pip install neptune")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.1/163.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 89.9 MB/s eta 0:00:00
Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 100.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818

   SECTION 3b: VERIFYING IMPORTS WITH PYTHON 3.12 PATCHES
Successfully injected dummy 'imp' module into sys.modules.
Patched pkgutil.ImpImporter missing attribute.
Injected compatibility stub for NeptuneLogger into pytorch_lightning.loggers.
------------------------------------------------------------
PyTorch          : 2.8.0+cu126
torchmetrics     : 1.9.0
pytorch-lightning: 2.6.6


NeMo             : 3.0.0
NeMo ASR         : OK
transformers     : 4.57.0
huggingface_hub  : 0.35.3
datasets         : 4.0.0
All imports verified successfully.


## Section 1: Environment Verification


In [2]:
import os, sys, shutil, platform, torch
import numpy as np

print("=" * 60)
print("         SECTION 1: ENVIRONMENT VERIFICATION")
print("=" * 60)

print(f"Python       : {platform.python_version()}")
print(f"NumPy        : {np.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Set Runtime -> Change runtime type -> T4 GPU")

print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"CUDA version : {torch.version.cuda}")
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU VRAM     : {vram:.1f} GB")
_, _, free = shutil.disk_usage("/")
print(f"Disk free    : {free / 1e9:.1f} GB")
print("=" * 60)


         SECTION 1: ENVIRONMENT VERIFICATION
Python       : 3.12.12
NumPy        : 2.0.2
PyTorch      : 2.8.0+cu126
CUDA avail   : True
GPU          : Tesla T4
CUDA version : 12.6
GPU VRAM     : 15.6 GB
Disk free    : 70.6 GB


## Section 2: Clone AI4Bharat NeMo (nemo-v2 branch)


In [3]:
import os, sys, subprocess

NEMO_DIR = "/content/NeMo"

if not os.path.isdir(NEMO_DIR):
    print("Cloning AI4Bharat/NeMo (branch: nemo-v2)...")
    r = subprocess.run(
        ["git", "clone", "-b", "nemo-v2",
         "https://github.com/AI4Bharat/NeMo.git", NEMO_DIR],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("Clone failed")
    print("Done.")
else:
    print(f"Already exists: {NEMO_DIR}")

branch = subprocess.check_output(
    ["git", "-C", NEMO_DIR, "rev-parse", "--abbrev-ref", "HEAD"]
).decode().strip()
commit = subprocess.check_output(
    ["git", "-C", NEMO_DIR, "log", "-1", "--oneline"]
).decode().strip()
print(f"Branch : {branch}")
print(f"Commit : {commit}")

if NEMO_DIR not in sys.path:
    sys.path.insert(0, NEMO_DIR)
    print(f"Added to sys.path")


Already exists: /content/NeMo
Branch : nemo-v2
Commit : 8dce88cf8 added shuffling


## Section 3a: Install Dependencies

This cell installs **pinned** versions of all NeMo dependencies.

> **After this cell finishes, you MUST restart the runtime:**
> `Runtime` → `Restart session`
>
> Then skip directly to **Section 3b** (do NOT re-run 3a).

### Why restart is required
Colab pre-loads `torchmetrics`, `pytorch_lightning`, and `lightning_fabric` into memory at startup. Even after pip replaces the files on disk, Python's import cache still holds the old (incompatible) versions. A runtime restart is the only reliable way to clear compiled C-extension caches.


In [4]:
import sys, subprocess

print("=" * 60)
print("   SECTION 3a: INSTALLING PINNED DEPENDENCIES")
print("=" * 60)

# Pinned package list — rationale for each pin:
#   setuptools>=70.0      Fixes pkg_resources for Python 3.12
#                         (pkgutil.ImpImporter was removed in 3.12)
#   lightning==2.2.5      Unified pkg: locks pytorch_lightning + lightning_fabric
#                         to the SAME version (prevents _TORCH_GREATER_EQUAL_2_1 error)
#   torchmetrics==1.3.2   Last version with _reduce_stat_scores in stat_scores.py
#                         (needed by dice.py, eagerly imported by PL)
#   hydra-core<=1.3.2     Required by AI4Bharat NeMo nemo-v2
#   omegaconf<=2.3        Required by AI4Bharat NeMo nemo-v2

pkgs = [
    "setuptools>=70.0",
    "lightning==2.2.5",
    "torchmetrics==1.3.2",
    "omegaconf>=2.2.0,<=2.3",
    "hydra-core>1.3,<=1.3.2",
    "soundfile",
    "sox",
    "sentencepiece",
    "editdistance",
    "jiwer",
    "webdataset>=0.2.86",
    "lhotse>=1.20.0",
]

cmd = [sys.executable, "-m", "pip", "install", "-q"] + pkgs
print(f"Running: pip install {' '.join(pkgs[:4])} ...")
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode != 0:
    print("STDOUT:", r.stdout[-3000:])
    print("STDERR:", r.stderr[-3000:])
    raise RuntimeError("pip install failed")
print("pip install: OK")

# Install NeMo editable (--no-deps to not touch PyTorch/CUDA)
nemo_cmd = [sys.executable, "-m", "pip", "install", "--no-deps", "-e", "/content/NeMo"]
r2 = subprocess.run(nemo_cmd, capture_output=True, text=True)
if r2.returncode != 0:
    print("STDERR:", r2.stderr[-2000:])
    raise RuntimeError("NeMo install failed")
print("NeMo install: OK")

# ── Patch lightning_fabric + lightning for Python 3.12 ────────
# These packages call pkg_resources.declare_namespace() in their
# __init__.py, which triggers Colab's SYSTEM pkg_resources at
# /usr/lib/python3/dist-packages/pkg_resources/__init__.py.
# That system copy is broken on Python 3.12 (uses removed APIs:
# pkgutil.ImpImporter, FileFinder.find_module).
# The declare_namespace() call is unnecessary for modern Python
# (PEP 420 implicit namespace packages), so we simply remove it.
import glob, site

site_pkgs = site.getsitepackages()
if not site_pkgs:
    site_pkgs = ["/usr/local/lib/python3.12/dist-packages"]

patched = 0
for sp in site_pkgs:
    for pkg in ["lightning_fabric", "lightning"]:
        init_path = os.path.join(sp, pkg, "__init__.py")
        if os.path.isfile(init_path):
            with open(init_path, "r") as f:
                content = f.read()
            needle = '__import__("pkg_resources").declare_namespace(__name__)'
            if needle in content:
                content = content.replace(
                    needle,
                    "# [Patched for Python 3.12] declare_namespace removed"
                )
                with open(init_path, "w") as f:
                    f.write(content)
                patched += 1
                print(f"  Patched: {init_path}")

if patched:
    print(f"Patched {patched} file(s) for Python 3.12 compatibility.")
else:
    print("No patching needed (declare_namespace not found).")

print()
print("=" * 60)
print("  >>> NOW RESTART THE RUNTIME <<<")
print("  Runtime -> Restart session")
print("  Then run Section 3b (skip this cell)")
print("=" * 60)


   SECTION 3a: INSTALLING PINNED DEPENDENCIES
Running: pip install setuptools>=70.0 lightning==2.2.5 torchmetrics==1.3.2 omegaconf>=2.2.0,<=2.3 ...
pip install: OK
NeMo install: OK
No patching needed (declare_namespace not found).

  >>> NOW RESTART THE RUNTIME <<<
  Runtime -> Restart session
  Then run Section 3b (skip this cell)


## Section 3b: Verify Dependencies (run after restart)

Run this cell **after** restarting the runtime. It verifies all imports work correctly.


In [5]:
import sys, os

# Re-add NeMo to path after restart
NEMO_DIR = "/content/NeMo"
if NEMO_DIR not in sys.path:
    sys.path.insert(0, NEMO_DIR)

print("=" * 60)
print("   SECTION 3b: VERIFYING IMPORTS")
print("=" * 60)

import torch
print(f"PyTorch          : {torch.__version__}")

import torchmetrics
print(f"torchmetrics     : {torchmetrics.__version__}")

# ── Python 3.12 Compatibility Safeguards ──────────────────────
# 1. Stub removed pkgutil.ImpImporter
import pkgutil
if not hasattr(pkgutil, "ImpImporter"):
    pkgutil.ImpImporter = type("ImpImporter", (), {})

# 2. Patch pkg_resources.declare_namespace to be a no-op
try:
    import pkg_resources
    pkg_resources.declare_namespace = lambda name: None
except Exception:
    pass

# 3. Patch lightning_fabric/__init__.py on disk if not already patched
import site
for sp in ["/usr/local/lib/python3.12/dist-packages"] + getattr(site, "getsitepackages", lambda: [])():
    for pkg in ["lightning_fabric", "lightning"]:
        init_f = os.path.join(sp, pkg, "__init__.py")
        if os.path.isfile(init_f):
            try:
                with open(init_f, "r") as f:
                    c = f.read()
                needle = '__import__("pkg_resources").declare_namespace(__name__)'
                if needle in c:
                    with open(init_f, "w") as f:
                        f.write(c.replace(needle, "# [Patched for Python 3.12] declare_namespace removed"))
                    print(f"Patched on disk: {init_f}")
            except Exception:
                pass

import pytorch_lightning as pl
print(f"pytorch-lightning: {pl.__version__}")

import nemo
import nemo.collections.asr as nemo_asr
from nemo.collections.asr.models import ASRModel
print(f"NeMo             : {nemo.__version__}")
print(f"NeMo ASR         : OK")

import transformers, huggingface_hub, datasets
print(f"transformers     : {transformers.__version__}")
print(f"huggingface_hub  : {huggingface_hub.__version__}")
print(f"datasets         : {datasets.__version__}")

print("-" * 60)
print("All imports verified successfully.")
print("=" * 60)


   SECTION 3b: VERIFYING IMPORTS
PyTorch          : 2.8.0+cu126
torchmetrics     : 1.3.2
pytorch-lightning: 2.6.6
NeMo             : 1.23.0rc0
NeMo ASR         : OK
transformers     : 4.46.3
huggingface_hub  : 0.23.2
datasets         : 3.2.0
------------------------------------------------------------
All imports verified successfully.


## Section 4: Checkpoint Download & Verification

Downloads the official trainable IndicConformer 600M `.nemo` checkpoint (~2.35 GB) from `ai4bharat/IndicConformer` on Hugging Face.

> **Note on Gated Model**: `ai4bharat/IndicConformer` is a gated repository.
> 1. Visit [huggingface.co/ai4bharat/IndicConformer](https://huggingface.co/ai4bharat/IndicConformer) and accept the license terms (approval is automatic & instant).
> 2. Provide your Hugging Face token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (or add `HF_TOKEN` in Colab Secrets 🔑).


In [6]:
import os, shutil, tarfile, getpass
from huggingface_hub import hf_hub_download

print("=" * 60)
print("         SECTION 4: CHECKPOINT VERIFICATION")
print("=" * 60)

CHECKPOINT_PATH = "/content/IndicConformer.nemo"

# Search candidate locations
candidates = [
    CHECKPOINT_PATH,
    "/content/drive/MyDrive/IndicConformer.nemo",
    "/content/drive/MyDrive/SIH-2026/IndicConformer.nemo",
]

found = None
for p in candidates:
    if os.path.isfile(p) and os.path.getsize(p) > 2e9:
        found = p
        break

if found and found != CHECKPOINT_PATH:
    print(f"Found at: {found}")
    if not os.path.exists(CHECKPOINT_PATH):
        os.symlink(found, CHECKPOINT_PATH)
elif found:
    print(f"Already present: {found}")
else:
    # Resolve Hugging Face authentication for gated repository
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not hf_token:
        try:
            from huggingface_hub import get_token
            hf_token = get_token()
        except Exception:
            pass

    if not hf_token:
        print("=" * 60)
        print("ai4bharat/IndicConformer is a gated repository.")
        print("1. Visit https://huggingface.co/ai4bharat/IndicConformer & accept terms (instant)")
        print("2. Create a Read token at https://huggingface.co/settings/tokens")
        print("=" * 60)
        hf_token = getpass.getpass("Enter your Hugging Face Token: ").strip()
        if hf_token:
            os.environ["HF_TOKEN"] = hf_token

    print("Downloading from ai4bharat/IndicConformer (~2.35 GB)...")
    try:
        dl = hf_hub_download(
            repo_id="ai4bharat/IndicConformer",
            filename="IndicConformer.nemo",
            local_dir="/content",
            token=hf_token if hf_token else None
        )
    except Exception as e:
        if "401" in str(e) or "gated" in str(e).lower():
            print("\n" + "!" * 60)
            print("AUTHENTICATION / GATED REPO ERROR:")
            print("1. Go to https://huggingface.co/ai4bharat/IndicConformer")
            print("2. Click 'Agree and access repository' (approval is instant)")
            print("3. Ensure your token from https://huggingface.co/settings/tokens is valid")
            print("!" * 60 + "\n")
        raise

    if dl != CHECKPOINT_PATH and not os.path.exists(CHECKPOINT_PATH):
        os.rename(dl, CHECKPOINT_PATH)
    print("Download complete.")

size_gb = os.path.getsize(CHECKPOINT_PATH) / 1e9
print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Size       : {size_gb:.2f} GB")

if size_gb < 2.0:
    raise ValueError(f"Checkpoint too small ({size_gb:.2f} GB), expected ~2.35 GB")

# Verify it's a valid tar/nemo archive
with tarfile.open(CHECKPOINT_PATH, "r:*") as tar:
    names = tar.getnames()
    print(f"Archive OK : {len(names)} files")
    has_config = any("model_config" in n for n in names)
    print(f"Has config : {has_config}")
    if not has_config:
        raise ValueError("Invalid .nemo: missing model_config.yaml")

print("Checkpoint verified.")
print("=" * 60)


         SECTION 4: CHECKPOINT VERIFICATION


[NeMo W 2026-09-12 19:57:35 nemo_logging:393] /usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:99: UserWarning: 
    Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
    You are not authenticated with the Hugging Face Hub in this notebook.
    If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
      warnings.warn(
    


ai4bharat/IndicConformer is a gated repository.
1. Visit https://huggingface.co/ai4bharat/IndicConformer & accept terms (instant)
2. Create a Read token at https://huggingface.co/settings/tokens


IndicConformer.nemo:   0%|          | 0.00/2.52G [00:00<?, ?B/s]

Download complete.
Checkpoint : /content/IndicConformer.nemo
Size       : 2.52 GB
Archive OK : 69 files
Has config : True
Checkpoint verified.


## Section 5: Dataset Paths Configuration

Locates `train.tsv`, `dev.tsv`, `test.tsv`, and the audio `clips/` directory.
The **test set is strictly frozen** — never used for training or validation.


In [7]:
import os, sys, zipfile, shutil

# Mount Google Drive if in Colab
if os.path.isdir("/content"):
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            print("Mounting Google Drive...")
            drive.mount('/content/drive')
    except Exception as e:
        print(f"Drive mount info: {e}")

# Working directory setup
local_dir = r"C:\Users\Asus\OneDrive\Documents\SIH-2026\iTantra\speech_project"
if os.path.isdir(local_dir):
    os.chdir(local_dir)
elif os.path.isdir("/content"):
    os.chdir("/content")

print("=" * 60)
print("         SECTION 5: DATASET PATHS")
print("=" * 60)

# 0. Detect and extract clips.zip (especially from Google Drive /content/drive/MyDrive/)
target_extract = "/content/dataset" if os.path.isdir("/content") else "dataset"
os.makedirs(target_extract, exist_ok=True)

zip_candidates = [
    "/content/drive/MyDrive/clips.zip",
    "/content/drive/MyDrive/dataset.zip",
    "/content/clips.zip",
    "/content/dataset.zip",
    "clips.zip",
    "dataset.zip",
]

# Check if clips are already extracted
already_extracted = False
for check_d in ["/content/dataset/clips", "/content/dataset", "dataset/clips", "dataset"]:
    if os.path.isdir(check_d) and any(f.endswith(".mp3") for f in os.listdir(check_d)[:15]):
        print(f"Clips already extracted in: {check_d}")
        already_extracted = True
        break

if not already_extracted:
    for z in zip_candidates:
        if os.path.isfile(z):
            print(f"Found archive: {z}")
            print(f"Extracting to {target_extract} (this takes ~10-20 seconds)...")
            try:
                # Use fast system unzip in Colab/Linux if available, otherwise zipfile
                if os.name != 'nt' and shutil.which("unzip"):
                    import subprocess
                    subprocess.run(["unzip", "-q", "-o", z, "-d", target_extract], check=True)
                else:
                    with zipfile.ZipFile(z, 'r') as zf:
                        zf.extractall(target_extract)
                print("Extraction complete.")
            except Exception as ze:
                print(f"Extraction warning: {ze}")
            break

# 1. Search candidates for TSV files (train.tsv, dev.tsv, test.tsv)
tsv_search_dirs = [
    "/content/drive/MyDrive",
    "/content/drive/MyDrive/SIH-2026/iTantra/speech_project",
    "/content/drive/MyDrive/SIH-2026/iTantra",
    "/content/drive/MyDrive/dataset",
    "/content/drive/MyDrive/CommonVoiceHindi",
    "/content/speech_project",
    "/content/dataset",
    "/content",
    "C:/Users/Asus/OneDrive/Documents/SIH-2026/iTantra/speech_project",
    "C:/Users/Asus/OneDrive/Documents/SIH-2026/iTantra",
    "C:/Users/Asus/OneDrive/Documents/SIH-2026/iTantra/dataset",
    os.getcwd(),
    os.path.abspath(".."),
]

DATA_DIR = None
for d in tsv_search_dirs:
    if os.path.isdir(d) and os.path.isfile(os.path.join(d, "train.tsv")):
        DATA_DIR = os.path.abspath(d)
        break

# Recursive search in /content if still not found
if DATA_DIR is None and os.path.isdir("/content"):
    for root, dirs, files in os.walk("/content"):
        if any(skip in root for skip in [".git", "NeMo", ".ipynb_checkpoints"]):
            continue
        if "train.tsv" in files:
            DATA_DIR = os.path.abspath(root)
            break

if DATA_DIR is None:
    print("\n" + "!" * 60)
    print("DATASET NOT FOUND IN COLAB RUNTIME")
    print("!" * 60)
    print("Please make sure train.tsv, dev.tsv, and test.tsv are in /content or Google Drive (My Drive).")
    print("!" * 60 + "\n")
    raise FileNotFoundError("train.tsv not found.")

TRAIN_TSV = os.path.join(DATA_DIR, "train.tsv")
DEV_TSV   = os.path.join(DATA_DIR, "dev.tsv")
TEST_TSV  = os.path.join(DATA_DIR, "test.tsv")

# 2. Search candidates for audio clips directory (.mp3 files)
clips_search_dirs = [
    "/content/dataset/clips",
    "/content/dataset",
    "/content/clips",
    os.path.join(DATA_DIR, "clips"),
    os.path.join(DATA_DIR, "clips", "clips"),
    "/content/drive/MyDrive/clips",
    "/content/drive/MyDrive/dataset/clips",
    "C:/Users/Asus/OneDrive/Documents/SIH-2026/iTantra/dataset/clips",
    "C:/Users/Asus/OneDrive/Documents/SIH-2026/iTantra/clips",
    os.path.join(os.getcwd(), "dataset", "clips"),
    os.path.join(os.getcwd(), "clips"),
]

CLIPS_DIR = None
for cd in clips_search_dirs:
    if os.path.isdir(cd):
        sample_files = [f for f in os.listdir(cd)[:20] if f.endswith(".mp3")]
        if sample_files:
            CLIPS_DIR = os.path.abspath(cd)
            break

if CLIPS_DIR is None and os.path.isdir("/content"):
    for root, dirs, files in os.walk("/content"):
        if any(skip in root for skip in [".git", "NeMo", ".ipynb_checkpoints", "drive"]):
            continue
        if any(f.endswith(".mp3") for f in files[:10]):
            CLIPS_DIR = os.path.abspath(root)
            break

if CLIPS_DIR is None:
    raise FileNotFoundError("Audio clips directory with .mp3 files not found.")

print(f"DATA_DIR  : {DATA_DIR}")
print(f"TRAIN_TSV : {TRAIN_TSV} (exists: {os.path.exists(TRAIN_TSV)})")
print(f"DEV_TSV   : {DEV_TSV} (exists: {os.path.exists(DEV_TSV)})")
print(f"TEST_TSV  : {TEST_TSV} (exists: {os.path.exists(TEST_TSV)}) [FROZEN]")
print(f"CLIPS_DIR : {CLIPS_DIR}")

mp3_count = len([f for f in os.listdir(CLIPS_DIR) if f.endswith(".mp3")])
print(f"MP3 files : {mp3_count:,}")
print("=" * 60)


Mounted at /content/drive
         SECTION 5: DATASET PATHS
Found archive: /content/drive/MyDrive/clips.zip
Extracting to /content/dataset (this takes ~10-20 seconds)...
Extraction complete.
DATA_DIR  : /content/drive/MyDrive
TRAIN_TSV : /content/drive/MyDrive/train.tsv (exists: True)
DEV_TSV   : /content/drive/MyDrive/dev.tsv (exists: True)
TEST_TSV  : /content/drive/MyDrive/test.tsv (exists: True) [FROZEN]
CLIPS_DIR : /content/dataset/clips
MP3 files : 19,029


## Section 6: Dataset Validation & Health Checks


In [8]:
import csv, soundfile as sf

print("=" * 60)
print("         SECTION 6: DATASET HEALTH CHECKS")
print("=" * 60)

def validate_split(tsv_path, clips_dir, name):
    if not os.path.exists(tsv_path):
        print(f"  {name}: FILE NOT FOUND")
        return None

    total = 0; valid = 0; missing = 0; empty_txt = 0; dupes = 0
    total_dur = 0.0
    seen = set()

    with open(tsv_path, "r", encoding="utf-8") as f:
        non_empty = (l for l in f if l.strip())
        for row in csv.DictReader(non_empty, delimiter="\t"):
            total += 1
            fn = row.get("path", "").strip()
            txt = row.get("sentence", "").strip()

            if not txt:
                empty_txt += 1; continue
            if fn in seen:
                dupes += 1; continue
            seen.add(fn)

            audio_path = os.path.join(clips_dir, fn)
            if not os.path.exists(audio_path):
                missing += 1; continue

            try:
                info = sf.info(audio_path)
                total_dur += info.duration
                valid += 1
            except Exception:
                missing += 1

    print(f"  {name.upper():8s}  total={total:,}  valid={valid:,}  "
          f"missing={missing}  empty_txt={empty_txt}  dupes={dupes}  "
          f"hours={total_dur/3600:.2f}")
    return {"valid": valid, "hours": total_dur / 3600}

train_stats = validate_split(TRAIN_TSV, CLIPS_DIR, "train")
dev_stats   = validate_split(DEV_TSV, CLIPS_DIR, "dev")
# test_stats  = validate_split(TEST_TSV, CLIPS_DIR, "test")  # FROZEN

print("=" * 60)


         SECTION 6: DATASET HEALTH CHECKS
  TRAIN     total=4,896  valid=4,896  missing=0  empty_txt=0  dupes=0  hours=5.88
  DEV       total=2,814  valid=2,814  missing=0  empty_txt=0  dupes=0  hours=3.73


## Section 7: Manifest Generation

Creates NeMo JSON-lines manifests where each line is:
```json
{"audio_filepath": "/path/to/clip.mp3", "duration": 4.12, "text": "transcript"}
```
Also creates small **smoke test** manifests (32 train / 16 dev).


In [9]:
import os, csv, json, soundfile as sf

print("=" * 60)
print("         SECTION 7: MANIFEST GENERATION")
print("=" * 60)

MANIFEST_DIR = "/content/manifests" if os.path.isdir("/content") else os.path.join(os.path.dirname(DATA_DIR), "manifests")
os.makedirs(MANIFEST_DIR, exist_ok=True)

TRAIN_MANIFEST = os.path.join(MANIFEST_DIR, "train_manifest.json")
DEV_MANIFEST   = os.path.join(MANIFEST_DIR, "dev_manifest.json")
SMOKE_TRAIN    = os.path.join(MANIFEST_DIR, "smoke_train.json")
SMOKE_DEV      = os.path.join(MANIFEST_DIR, "smoke_dev.json")

def tsv_to_manifest(tsv_path, clips_dir, out_path, min_dur=0.2, max_dur=20.0, lang="hi"):
    entries = []
    seen = set()
    with open(tsv_path, "r", encoding="utf-8") as f:
        non_empty = (l for l in f if l.strip())
        for row in csv.DictReader(non_empty, delimiter="\t"):
            fn = row.get("path", "").strip()
            txt = row.get("sentence", "").strip()
            if not fn or not txt or fn in seen:
                continue
            seen.add(fn)

            ap = os.path.join(clips_dir, fn)
            if not os.path.exists(ap):
                continue
            try:
                info = sf.info(ap)
                dur = round(info.duration, 3)
                if dur < min_dur or dur > max_dur:
                    continue
                entries.append({
                    "audio_filepath": os.path.abspath(ap),
                    "duration": dur,
                    "text": txt,
                    "lang": lang,  # Required by IndicConformer's aggregate tokenizer
                })
            except Exception:
                continue

    with open(out_path, "w", encoding="utf-8") as out:
        for e in entries:
            out.write(json.dumps(e, ensure_ascii=False) + "\n")
    print(f"  {os.path.basename(out_path):30s} -> {len(entries):,} entries")
    return entries

train_entries = tsv_to_manifest(TRAIN_TSV, CLIPS_DIR, TRAIN_MANIFEST, lang="hi")
dev_entries   = tsv_to_manifest(DEV_TSV, CLIPS_DIR, DEV_MANIFEST, lang="hi")

# Smoke test manifests
def write_subset(entries, path, n):
    with open(path, "w", encoding="utf-8") as f:
        for e in entries[:n]:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")
    print(f"  {os.path.basename(path):30s} -> {min(n, len(entries))} entries")

write_subset(train_entries, SMOKE_TRAIN, 32)
write_subset(dev_entries, SMOKE_DEV, 16)

if train_entries:
    print(f"\nSample entry:")
    print(json.dumps(train_entries[0], indent=2, ensure_ascii=False))

print("=" * 60)


         SECTION 7: MANIFEST GENERATION
  train_manifest.json            -> 4,896 entries
  dev_manifest.json              -> 2,814 entries
  smoke_train.json               -> 32 entries
  smoke_dev.json                 -> 16 entries

Sample entry:
{
  "audio_filepath": "/content/dataset/clips/common_voice_hi_27478478.mp3",
  "duration": 4.457,
  "text": "दिल्ली महिला आयोग को डरा धमका रही है दिल्ली पुलिस: स्वाति मालीवाल",
  "lang": "hi"
}


## Section 8: Model Loading & Configuration

Loads the IndicConformer 600M hybrid CTC/RNNT checkpoint.

**Training strategy for Hindi adaptation on T4 (16 GB VRAM):**
- Switch to **CTC decoding** (simpler to validate)
- **Freeze RNNT decoder & joint heads** (saves ~30% VRAM)
- **Keep encoder + CTC decoder trainable**
- Low learning rate (2e-5) to preserve multilingual knowledge


In [10]:
import torch
from omegaconf import OmegaConf, open_dict
from nemo.collections.asr.models import ASRModel

print("=" * 60)
print("         SECTION 8: MODEL LOADING")
print("=" * 60)

print(f"Restoring from: {CHECKPOINT_PATH}")
asr_model = ASRModel.restore_from(CHECKPOINT_PATH, map_location="cpu")
print(f"Model class  : {asr_model.__class__.__name__}")

asr_model = asr_model.cuda()

total_params = sum(p.numel() for p in asr_model.parameters())
print(f"Total params : {total_params / 1e6:.1f}M")

# Switch to CTC decoding
has_ctc = hasattr(asr_model, "ctc_decoder") and asr_model.ctc_decoder is not None
has_rnnt = hasattr(asr_model, "decoder") and asr_model.decoder is not None
print(f"Has CTC      : {has_ctc}")
print(f"Has RNNT     : {has_rnnt}")

if has_ctc:
    asr_model.change_decoding_strategy(decoder_type="ctc")
    print("Decoding     : CTC")

# Freeze RNNT decoder + joint to save VRAM
if has_rnnt:
    for p in asr_model.decoder.parameters():
        p.requires_grad = False
    if hasattr(asr_model, "joint") and asr_model.joint is not None:
        for p in asr_model.joint.parameters():
            p.requires_grad = False
    print("Frozen       : RNNT decoder + joint")

trainable = sum(p.numel() for p in asr_model.parameters() if p.requires_grad)
print(f"Trainable    : {trainable / 1e6:.1f}M")
print("=" * 60)


         SECTION 8: MODEL LOADING
Restoring from: /content/IndicConformer.nemo
[NeMo I 2026-09-12 20:02:05 nemo_logging:381] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-09-12 20:02:05 nemo_logging:381] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-12 20:02:05 nemo_logging:381] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-12 20:02:05 nemo_logging:381] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-12 20:02:05 nemo_logging:381] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-12 20:02:05 nemo_logging:381] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-12 20:02:05 nemo_logging:381] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-12 20:02:05 nemo_logging:381] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2026-09-12 20:02:05 nemo_logging:381] Tokenizer SentencePieceTokeniz

[NeMo W 2026-09-12 20:02:11 nemo_logging:393] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    sample_rate: 16000
    batch_size: 22
    shuffle: true
    num_workers: 16
    pin_memory: true
    max_duration: 30.0
    min_duration: 0.2
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    is_concat: true
    concat_sampling_technique: temperature
    concat_sampling_temperature: 1000000000.0
    return_language_id: true
    manifest_filepath:
    - /projects/data/asrteam/speechteam/speech-datasets/manifests/nemo/vistaarv4.7.1/train_assamese.json
    - /projects/data/asrteam/speechteam/speech-datasets/manifests/nemo/vistaarv4.7.1/train_bengali.json
    - /projects/data/asrteam/speechteam/speech-datasets/manifests/nemo/vistaarv4.7.1/train_bodo.json
 

[NeMo I 2026-09-12 20:02:11 nemo_logging:381] PADDING: 0
[NeMo I 2026-09-12 20:02:19 nemo_logging:381] Vocab size for each language: 256
[NeMo I 2026-09-12 20:02:19 nemo_logging:381] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-09-12 20:02:19 nemo_logging:381] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-09-12 20:02:21 nemo_logging:381] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-09-12 20:02:21 nemo_logging:381] Creating masks for multi-softmax layer.
[NeMo I 2026-09-12 20:02:21 nemo_logging:381] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-09-12 20:02:24 nemo_logging:381] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /content/IndicConformer.nemo.
Model class  : EncDecHybridRNNTC

## Section 9: Smoke Test

A tiny 5-step training run to verify the entire pipeline works:
dataloader, GPU forward/backward, loss computation, optimizer, validation.


In [12]:
import pytorch_lightning as pl
from omegaconf import open_dict, OmegaConf
import numpy as np

print("=" * 60)
print("         SECTION 9: SMOKE TEST")
print("=" * 60)

# NumPy 2.0 Patch
if not hasattr(np, "sctypes"):
    np.sctypes = {
        'int': [np.int8, np.int16, np.int32, np.int64],
        'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
        'float': [np.float16, np.float32, np.float64],
        'complex': [np.complex64, np.complex128],
        'others': [bool, object, bytes, str, np.void],
    }

# Set num_workers=0 to eliminate multiprocessing deadlocks in Colab
# Set num_workers=0 to eliminate multiprocessing deadlocks in Colab
smoke_train_cfg = {
    'manifest_filepath': SMOKE_TRAIN,
    'sample_rate': 16000,
    'batch_size': 2,
    'shuffle': True,
    'num_workers': 0,       # <-- 0 prevents Colab thread freezing
    'pin_memory': False,
    'max_duration': 16.7,
    'min_duration': 0.2,
    'return_sample_id': True,    # <-- Required for multilingual unpacking (6 items)
    'return_language_id': True,  # <-- Required for multilingual unpacking (6 items)
}

smoke_val_cfg = {
    'manifest_filepath': SMOKE_DEV,
    'sample_rate': 16000,
    'batch_size': 2,
    'shuffle': False,
    'num_workers': 0,       # <-- 0 prevents Colab thread freezing
    'pin_memory': False,
    'return_sample_id': True,    # <-- Required for validation_pass
    'return_language_id': True,  # <-- Required for validation_pass
}

with open_dict(asr_model.cfg):
    asr_model.cfg.train_ds = OmegaConf.create(smoke_train_cfg)
    asr_model.cfg.validation_ds = OmegaConf.create(smoke_val_cfg)

asr_model.setup_training_data(asr_model.cfg.train_ds)
asr_model.setup_validation_data(asr_model.cfg.validation_ds)

print("Running smoke test...")
smoke_trainer.fit(asr_model)
print("-" * 60)
print("SMOKE TEST PASSED")
print("=" * 60)


         SECTION 9: SMOKE TEST
[NeMo I 2026-09-12 20:09:29 nemo_logging:381] Dataset loaded with 32 files totalling 0.03 hours
[NeMo I 2026-09-12 20:09:29 nemo_logging:381] 0 files were filtered totalling 0.00 hours
[NeMo I 2026-09-12 20:09:30 nemo_logging:381] Dataset loaded with 16 files totalling 0.02 hours
[NeMo I 2026-09-12 20:09:30 nemo_logging:381] 0 files were filtered totalling 0.00 hours


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Running smoke test...
[NeMo I 2026-09-12 20:09:31 nemo_logging:381] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: [0.9, 0.98]
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 2.357
        maximize: False
        weight_decay: 0.001
    )
[NeMo I 2026-09-12 20:09:31 nemo_logging:381] Scheduler "<nemo.core.optim.lr_scheduler.NoamAnnealing object at 0x7ac691812ea0>" 
    will be used during training (effective maximum steps = 5) - 
    Parameters : 
    (d_model: 1024
    warmup_steps: 21703
    warmup_ratio: null
    min_lr: 1.0e-06
    max_steps: 5
    )


[NeMo W 2026-09-12 20:09:31 nemo_logging:393] /usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.
    


┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type                              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ preprocessor      │ AudioToMelSpectrogramPreprocessor │      0 │ train │     0 │
│ 1 │ encoder           │ ConformerEncoder                  │  607 M │ train │     0 │
│ 2 │ decoder           │ RNNTDecoder                       │ 10.2 M │ train │     0 │
│ 3 │ joint             │ RNNTJoint                         │  4.7 M │ train │     0 │
│ 4 │ loss              │ RNNTLoss                          │      0 │ train │     0 │
│ 5 │ spec_augmentation │ SpectrogramAugmentation           │      0 │ train │     0 │
│ 6 │ wer               │ WER                               │      0 │ train │     0 │
│ 7 │ ctc_decoder       │ ConvASRDecoder                    │  5.8 M │ train │     0 │
│ 8 │ ctc_loss          │ CTCLoss                           │      0 │ train │     0 │
│ 9 │ ctc_wer           │ WER                               │      0 │ train │     0 │
└───┴───────────────────┴───────────────────────────────────┴────────┴───────┴───────┘

...

[NeMo I 2026-09-12 20:09:48 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:09:48 nemo_logging:381] reference: तुम्हारी उम्र के लोगों को अक्सर इस तरह की दिक्कत होती है<unk>
[NeMo I 2026-09-12 20:09:48 nemo_logging:381] predicted: तुम्हारी उम्र के लोगों को अक्सर इस तरह की दिक्कत होती है
[NeMo I 2026-09-12 20:09:48 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:09:48 nemo_logging:381] reference: तुम्हारी उम्र के लोगों को अक्सर इस तरह की दिक्कत होती है<unk>
[NeMo I 2026-09-12 20:09:48 nemo_logging:381] predicted: तुम्हारी उम्र के लोगों को अक्सर इस तरह की दिक्कत होती है


[NeMo W 2026-09-12 20:09:48 nemo_logging:393] /usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 2. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
    


[NeMo I 2026-09-12 20:09:49 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:09:49 nemo_logging:381] reference: तब तक मुझे क्या करना चाहिए<unk>
[NeMo I 2026-09-12 20:09:49 nemo_logging:381] predicted: तब तक मुझे क्या करना चाहिए
[NeMo I 2026-09-12 20:09:49 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:09:49 nemo_logging:381] reference: तब तक मुझे क्या करना चाहिए<unk>
[NeMo I 2026-09-12 20:09:49 nemo_logging:381] predicted: तब तक मुझे क्या करना चाहिए


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=5` reached.


------------------------------------------------------------
SMOKE TEST PASSED


## Section 10: Full Fine-Tuning

Hindi CTC adaptation with:
- **Batch size:** 4, gradient accumulation 4x (effective 16)
- **Optimizer:** AdamW, lr=2e-5
- **Precision:** 16-mixed (AMP FP16)
- **Epochs:** 5
- **Checkpointing:** Best val_loss saved


In [13]:
import os
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from omegaconf import open_dict, OmegaConf

print("=" * 60)
print("         SECTION 10: FULL FINE-TUNING")
print("=" * 60)

OUTPUT_DIR = "/content/outputs/indicconformer_hi_finetuned" if os.path.isdir("/content") else os.path.join(os.path.dirname(DATA_DIR), "outputs", "indicconformer_hi_finetuned")
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

train_cfg = {
    'manifest_filepath': TRAIN_MANIFEST,
    'sample_rate': 16000,
    'batch_size': 4,
    'shuffle': True,
    'num_workers': 2,
    'pin_memory': True,
    'max_duration': 16.7,
    'min_duration': 0.2,
    'return_sample_id': True,
    'return_language_id': True,
}
val_cfg = {
    'manifest_filepath': DEV_MANIFEST,
    'sample_rate': 16000,
    'batch_size': 4,
    'shuffle': False,
    'num_workers': 2,
    'pin_memory': True,
    'return_sample_id': True,
    'return_language_id': True,
}


with open_dict(asr_model.cfg):
    asr_model.cfg.train_ds = OmegaConf.create(train_cfg)
    asr_model.cfg.validation_ds = OmegaConf.create(val_cfg)

asr_model.setup_training_data(asr_model.cfg.train_ds)
asr_model.setup_validation_data(asr_model.cfg.validation_ds)

# Optimizer
with open_dict(asr_model.cfg):
    if not hasattr(asr_model.cfg, "optim"):
        asr_model.cfg.optim = OmegaConf.create({})
    asr_model.cfg.optim.name = "adamw"
    asr_model.cfg.optim.lr = 2e-5
    asr_model.cfg.optim.betas = [0.9, 0.98]
    asr_model.cfg.optim.weight_decay = 1e-3

asr_model.setup_optimization(asr_model.cfg.optim)

ckpt_cb = ModelCheckpoint(
    dirpath=CKPT_DIR,
    filename="indicconformer-hi-{epoch:02d}-{val_loss:.3f}",
    save_top_k=2,
    monitor="val_loss",
    mode="min",
    save_last=True,
)
lr_cb = LearningRateMonitor(logging_interval="step")

NUM_EPOCHS = 5

trainer = pl.Trainer(
    devices=1,
    accelerator="gpu",
    max_epochs=NUM_EPOCHS,
    accumulate_grad_batches=4,
    precision="16-mixed",
    callbacks=[ckpt_cb, lr_cb],
    enable_checkpointing=True,
    log_every_n_steps=10,
    val_check_interval=1.0,
)

print(f"Training for {NUM_EPOCHS} epochs...")
trainer.fit(asr_model)
print("-" * 60)
print("Fine-tuning complete.")
print("=" * 60)


         SECTION 10: FULL FINE-TUNING
[NeMo I 2026-09-12 20:10:20 nemo_logging:381] Dataset loaded with 4896 files totalling 5.88 hours
[NeMo I 2026-09-12 20:10:20 nemo_logging:381] 0 files were filtered totalling 0.00 hours
[NeMo I 2026-09-12 20:10:22 nemo_logging:381] Dataset loaded with 2814 files totalling 3.73 hours
[NeMo I 2026-09-12 20:10:22 nemo_logging:381] 0 files were filtered totalling 0.00 hours
[NeMo I 2026-09-12 20:10:22 nemo_logging:381] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: [0.9, 0.98]
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 2e-05
        maximize: False
        weight_decay: 0.001
    )
[NeMo I 2026-09-12 20:10:22 nemo_logging:381] Scheduler "<nemo.core.optim.lr_scheduler.NoamAnnealing object at 0x7ac691b23710>" 
    will be used during training (effective maximum steps = 5) - 
    Parameters : 


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..


Training for 5 epochs...


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[NeMo I 2026-09-12 20:10:40 nemo_logging:381] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: [0.9, 0.98]
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 2e-05
        maximize: False
        weight_decay: 0.001
    )
[NeMo I 2026-09-12 20:10:40 nemo_logging:381] Scheduler "<nemo.core.optim.lr_scheduler.NoamAnnealing object at 0x7ac62a0a1ac0>" 
    will be used during training (effective maximum steps = 1530) - 
    Parameters : 
    (d_model: 1024
    warmup_steps: 21703
    warmup_ratio: null
    min_lr: 1.0e-06
    max_steps: 1530
    )


...

[NeMo I 2026-09-12 20:28:01 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:28:01 nemo_logging:381] reference: नौकरी का झांसा देकर लड़कियों को बनाया बंधक<unk> पुलिस ने बचाया
[NeMo I 2026-09-12 20:28:01 nemo_logging:381] predicted: नौकरी का झांसा देकर लड़कियों को बनाया बंधक पुलिस ने बचाया
[NeMo I 2026-09-12 20:28:01 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:28:01 nemo_logging:381] reference: नौकरी का झांसा देकर लड़कियों को बनाया बंधक<unk> पुलिस ने बचाया
[NeMo I 2026-09-12 20:28:01 nemo_logging:381] predicted: नौकरी का सा देकर लड़कियों को बनाया बंधक पुलिस ने बचाया


[NeMo I 2026-09-12 20:28:01 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:28:01 nemo_logging:381] reference: एक्सक्लूसिव<unk> बंदरगाह पर पड़ी है लाखों टन चीनी
[NeMo I 2026-09-12 20:28:01 nemo_logging:381] predicted: एक्सक्लूसिव बंदरगाह पर बड़ी है लाखों टन चीनी
[NeMo I 2026-09-12 20:28:01 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:28:01 nemo_logging:381] reference: एक्सक्लूसिव<unk> बंदरगाह पर पड़ी है लाखों टन चीनी
[NeMo I 2026-09-12 20:28:01 nemo_logging:381] predicted: एक्सक्लूसिव बंदरगाह पर बड़ी है लाखों टन चीनी


[NeMo I 2026-09-12 20:28:02 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:28:02 nemo_logging:381] reference: मुझे और एक कप चाय दे दो<unk>
[NeMo I 2026-09-12 20:28:02 nemo_logging:381] predicted: मुझे और एक कप चाय दे दो
[NeMo I 2026-09-12 20:28:02 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:28:02 nemo_logging:381] reference: मुझे और एक कप चाय दे दो<unk>
[NeMo I 2026-09-12 20:28:02 nemo_logging:381] predicted: मुझे और एक कप चाय दे दो


[NeMo I 2026-09-12 20:28:03 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:28:03 nemo_logging:381] reference: उसके पास बहुत सारी इतिहास की किताबें हैं<unk>
[NeMo I 2026-09-12 20:28:03 nemo_logging:381] predicted: उसके पास बहुत सारी इतिहास की किताबें हैं
[NeMo I 2026-09-12 20:28:03 nemo_logging:381] 
    
[NeMo I 2026-09-12 20:28:03 nemo_logging:381] reference: उसके पास बहुत सारी इतिहास की किताबें हैं<unk>
[NeMo I 2026-09-12 20:28:03 nemo_logging:381] predicted: उसके पास बहुत सारी इतिहास की किताबें हैं


: 

## Section 11: Save Fine-Tuned Checkpoint


In [ ]:
import os
from omegaconf import OmegaConf

print("=" * 60)
print("         SECTION 11: CHECKPOINT SAVING")
print("=" * 60)

FINAL_NEMO = os.path.join(OUTPUT_DIR, "indicconformer_hi_finetuned.nemo")
CFG_PATH   = os.path.join(OUTPUT_DIR, "model_config.yaml")

print(f"Saving .nemo to: {FINAL_NEMO}")
asr_model.save_to(FINAL_NEMO)
OmegaConf.save(asr_model.cfg, CFG_PATH)

orig_gb = os.path.getsize(CHECKPOINT_PATH) / 1e9
new_gb  = os.path.getsize(FINAL_NEMO) / 1e9
print(f"Original : {orig_gb:.2f} GB (untouched)")
print(f"Fine-tuned: {new_gb:.2f} GB")
print("=" * 60)


## Section 12: Frozen Test-Set Evaluation

Evaluates on the **exact same 3,342-sample test set** used for the 16.72% baseline.
Uses identical text normalization and Levenshtein WER computation.


In [ ]:
import os, sys, time, csv, re
import numpy as np
import soundfile as sf
import scipy.signal
import torch
from nemo.collections.asr.models import ASRModel

print("=" * 60)
print("     SECTION 12: FROZEN TEST-SET EVALUATION")
print("=" * 60)

# ── Text normalization (identical to evaluate_stt.py) ────────
def normalize_text(text):
    text = str(text).strip().lower()
    text = re.sub(r"[^\w\s\u0900-\u097F]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ── Levenshtein edit distance (word-level) ───────────────────
def edit_distance(ref_words, hyp_words):
    n, m = len(ref_words), len(hyp_words)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            c = 0 if ref_words[i-1] == hyp_words[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+c)
    return dp[n][m]

# ── Load fine-tuned model ────────────────────────────────────
print("Loading fine-tuned model...")
eval_model = ASRModel.restore_from(FINAL_NEMO).cuda().eval()
if hasattr(eval_model, "change_decoding_strategy"):
    eval_model.change_decoding_strategy(decoder_type="ctc")

# ── Read frozen test.tsv ─────────────────────────────────────
test_rows = []
with open(TEST_TSV, "r", encoding="utf-8") as f:
    for row in csv.DictReader(f, delimiter="\t"):
        p = row.get("path", "").strip()
        s = row.get("sentence", "").strip()
        if p and s:
            test_rows.append({"filename": p, "sentence": s})
print(f"Test samples: {len(test_rows):,}")

# ── Evaluation loop ──────────────────────────────────────────
results = []
total_audio_sec = 0.0
total_inf_sec = 0.0

for idx, item in enumerate(test_rows, 1):
    fn = item["filename"]
    ref = item["sentence"]
    ap = os.path.join(CLIPS_DIR, fn)
    if not os.path.exists(ap):
        continue

    try:
        audio, sr = sf.read(ap)
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)
        audio = audio.astype(np.float32)
        if sr != 16000:
            new_len = int(len(audio) * 16000 / sr)
            audio = scipy.signal.resample(audio, new_len).astype(np.float32)
            sr = 16000
        dur = len(audio) / sr

        t0 = time.perf_counter()
        with torch.inference_mode():
            try:
                out = eval_model.transcribe([ap], batch_size=1,
                                            verbose=False, language_id="hi")
            except TypeError:
                out = eval_model.transcribe([ap], batch_size=1, verbose=False)
        inf_time = time.perf_counter() - t0

        # Extract hypothesis text
        if isinstance(out, tuple):
            out = out[0]
        if isinstance(out, list) and out:
            hyp = out[0].text if hasattr(out[0], "text") else str(out[0])
        else:
            hyp = str(out)
        hyp = hyp.strip()

        wer_val = edit_distance(
            normalize_text(ref).split(),
            normalize_text(hyp).split()
        ) / max(len(normalize_text(ref).split()), 1)

        rtf = inf_time / dur if dur > 0 else 0
        total_audio_sec += dur
        total_inf_sec += inf_time

        results.append({
            "filename": fn, "reference": ref, "prediction": hyp,
            "wer": wer_val, "audio_duration": dur,
            "inference_time": inf_time, "rtf": rtf
        })

        if idx % 500 == 0 or idx == len(test_rows):
            print(f"  [{idx}/{len(test_rows)}] wer={wer_val*100:.1f}%  rtf={rtf:.3f}")

    except Exception as e:
        print(f"  Error on {fn}: {e}")

# ── Corpus-level WER ─────────────────────────────────────────
total_ref_words = 0
total_edits = 0
for r in results:
    rw = normalize_text(r["reference"]).split()
    hw = normalize_text(r["prediction"]).split()
    total_ref_words += len(rw)
    total_edits += edit_distance(rw, hw)

corpus_wer = total_edits / total_ref_words if total_ref_words > 0 else 0
avg_latency = total_inf_sec / len(results) if results else 0
avg_rtf = total_inf_sec / total_audio_sec if total_audio_sec > 0 else 0

# ── Save results CSV ─────────────────────────────────────────
eval_csv = os.path.join(OUTPUT_DIR, "stt_finetuned_results.csv")
with open(eval_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["filename","reference","prediction","wer",
                "audio_duration","inference_time","rtf"])
    for r in results:
        w.writerow([r["filename"], r["reference"], r["prediction"],
                    f"{r['wer']:.6f}", f"{r['audio_duration']:.4f}",
                    f"{r['inference_time']:.4f}", f"{r['rtf']:.6f}"])

print(f"\nResults saved to: {eval_csv}")
print(f"Evaluated : {len(results):,} samples")
print(f"Corpus WER: {corpus_wer*100:.2f}%")
print(f"Avg latency: {avg_latency:.3f}s")
print(f"Avg RTF    : {avg_rtf:.4f}")
print("=" * 60)


## Section 13: Baseline vs Fine-Tuned Comparison


In [ ]:
import json

print("=" * 60)
print("     SECTION 13: BASELINE vs FINE-TUNED")
print("=" * 60)

BL_WER = 16.72
BL_LAT = 0.288
BL_RTF = 0.056
BL_N   = 3342

ft_wer = corpus_wer * 100
wer_diff = BL_WER - ft_wer
rel_impr = (wer_diff / BL_WER) * 100

print(f"{'Metric':<20} {'Baseline':>12} {'Fine-tuned':>12} {'Delta':>12}")
print("-" * 58)
print(f"{'Corpus WER':<20} {BL_WER:>11.2f}% {ft_wer:>11.2f}% {wer_diff:>+11.2f}%")
print(f"{'  (relative)':<20} {'':>12} {'':>12} {rel_impr:>+10.1f}% rel")
print(f"{'Avg Latency':<20} {BL_LAT:>11.3f}s {avg_latency:>11.3f}s {avg_latency-BL_LAT:>+11.3f}s")
print(f"{'Avg RTF':<20} {BL_RTF:>11.4f} {avg_rtf:>11.4f} {avg_rtf-BL_RTF:>+11.4f}")
print(f"{'Test Samples':<20} {BL_N:>12,} {len(results):>12,}")
print("-" * 58)

summary = {
    "baseline": {"wer": BL_WER, "latency": BL_LAT, "rtf": BL_RTF, "samples": BL_N},
    "finetuned": {"wer": round(ft_wer,2), "latency": round(avg_latency,4),
                  "rtf": round(avg_rtf,4), "samples": len(results)},
    "improvement": {"abs_wer_points": round(wer_diff,2),
                    "rel_wer_pct": round(rel_impr,2)}
}
summary_path = os.path.join(OUTPUT_DIR, "benchmark_comparison.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved to: {summary_path}")
print("=" * 60)


## Section 14: Qualitative Inspection & Next Steps


In [ ]:
print("=" * 60)
print("     SECTION 14: SAMPLE RESULTS")
print("=" * 60)

for i, s in enumerate(results[:5], 1):
    print(f"\nSample #{i}: {s['filename']}")
    print(f"  REF : {s['reference']}")
    print(f"  HYP : {s['prediction']}")
    print(f"  WER : {s['wer']*100:.1f}%  RTF: {s['rtf']:.3f}")
    print("-" * 60)

roadmap = (
    "\nNEXT STEPS (Android Deployment Roadmap):\n"
    "1. Export fine-tuned Conformer encoder + CTC decoder to ONNX\n"
    "2. INT8 quantization (reduce ~2.35 GB -> ~600 MB)\n"
    "3. Deploy via ONNX Runtime Mobile or Sherpa-ONNX on Android\n"
    "4. Chain with IndicTrans2 (MT) + IndicF5/FastSpeech2 (TTS)\n"
)
print(roadmap)
print("=" * 60)
print("Pipeline complete.")
print("=" * 60)
